# LeetCode SQL → pandas · Bagian Easy **tanpa `pd.merge`**

Notebook ini menyelesaikan sembilan soal Easy yang sama seperti notebook utama, tetapi dengan satu batasan yang sengaja dipasang: **`pd.merge`, `DataFrame.join`, dan `pd.concat` tidak boleh dipakai.**

Batasan ini bukan latihan gaya-gayaan. Tujuannya satu:

> **`JOIN` bukan operasi dasar. Ia adalah nama untuk beberapa operasi berbeda yang kebetulan ditulis dengan kata kunci yang sama di SQL.**

Selama `merge` selalu tersedia, perbedaan itu tidak pernah terasa — semuanya jadi "ya, di-join saja". Begitu `merge` dicabut, Anda terpaksa bertanya: *sebenarnya operasi apa yang saya butuhkan di sini?* Dan jawabannya ternyata cuma tiga:

| Yang sebenarnya Anda mau | Alat di pandas | Contoh |
|---|---|---|
| **Lookup** — ambil nilai dari tabel referensi | `Series.map()` | E4, E7 |
| **Uji keanggotaan** — apakah kunci ini ada di sana? | `Series.isin()` | E5 |
| **Reshape** — pindahkan nilai dari baris ke kolom | `.unstack()` / `.pivot_table()` | E9 |

Enam soal sisanya (E1, E2, E3, E6, E8) memang tidak pernah butuh join sama sekali — itu sendiri sudah menjadi pelajaran: refleks "join dulu" sering kali muncul lebih cepat daripada kebutuhannya.

> **Prasyarat:** notebook ini berdiri sendiri. Tiap sel kode membuat datanya sendiri, jadi bisa dijalankan dari mana saja setelah sel setup.

## Peta terjemahan: `JOIN` tanpa `merge`

| SQL | `merge` (notebook utama) | Tanpa `merge` (notebook ini) |
|---|---|---|
| `LEFT JOIN` satu kolom, kunci kanan unik | `l.merge(r, on="k", how="left")` | `l["k"].map(r.set_index("k")["v"])` |
| `LEFT JOIN` beberapa kolom sekaligus | `l.merge(r, on="k", how="left")` | `r.set_index("k").reindex(l["k"]).reset_index()` |
| `WHERE k IN (SELECT ...)` (semi-join) | `merge(how="inner")` | `l[l["k"].isin(r["k"])]` |
| `WHERE k NOT IN (SELECT ...)` (anti-join) | `merge(how="left")` + `.isna()` | `l[~l["k"].isin(r["k"])]` |
| `LEFT JOIN` + `COUNT` (kunci kanan **tidak** unik) | `merge` lalu `groupby` | agregasi dulu → `map` |
| self-join berurutan (`LAG`) | `merge` tabel ke dirinya sendiri | `.sort_values()` + `.shift()` |
| `MAX(CASE WHEN ...)` per grup | — | `.unstack()` / `.pivot_table()` |

**Satu keuntungan nyata dari `map`, bukan sekadar selera.** Kalau kunci di sisi kanan ternyata **tidak unik**, `merge` akan diam-diam menggandakan baris di sisi kiri dan laporan Anda membengkak tanpa peringatan. `map` justru **melempar error**. Gagal berisik jauh lebih murah daripada gagal diam-diam — ini didemonstrasikan langsung di E4.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("pandas", pd.__version__)
print("numpy ", np.__version__)


def show(judul, df):
    """Cetak judul lalu tampilkan DataFrame/Series/skalar."""
    print(f"=== {judul} ===")
    if isinstance(df, (pd.DataFrame, pd.Series)):
        display(df)
    else:
        print(df)
    print()


# Pagar pengaman: memanggil merge/join/concat di notebook ini akan langsung berhenti.
_merge_asli = pd.merge

def _dilarang(*args, **kwargs):
    raise RuntimeError("pd.merge dilarang di notebook ini - pakai map / isin / unstack.")

pd.merge = _dilarang
pd.DataFrame.merge = _dilarang
pd.DataFrame.join = _dilarang

print("\npagar aktif: pd.merge, DataFrame.merge, DataFrame.join dinonaktifkan.")

pandas 3.0.5
numpy  2.5.1

pagar aktif: pd.merge, DataFrame.merge, DataFrame.join dinonaktifkan.


---
### E1 · LC 1757 — Recyclable and Low Fat Products · `Easy`

**Butuh join?** Tidak — satu tabel saja.

**Tugas.** `Products(product_id, low_fats, recyclable)` berisi flag `'Y'`/`'N'`. Ambil `product_id` yang kedua flag-nya `'Y'`.

```sql
SELECT product_id FROM Products WHERE low_fats = 'Y' AND recyclable = 'Y';
```

**Jebakan.** Operator logika pada Series adalah `&`, `|`, `~` — bukan `and`, `or`, `not`. Tiap kondisi wajib dikurung, karena `&` mengikat lebih kuat daripada `==`.

In [27]:
products = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "low_fats":   ["Y", "Y", "N", "Y", "N"],
    "recyclable": ["N", "Y", "Y", "Y", "N"],
})

mask = (products["low_fats"] == "Y") & (products["recyclable"] == "Y")

In [28]:
mask

0    False
1     True
2    False
3     True
4    False
dtype: bool

In [30]:
hasil = products.loc[mask, ["product_id"]]
show("E1 — hasil", hasil)

assert hasil["product_id"].tolist() == [2, 4]

=== E1 — hasil ===


,product_id
1,2
3,4


### E2 · LC 584 — Find Customer Referee · `Easy`

**Butuh join?** Tidak.

**Tugas.** `Customer(id, name, referee_id)`. Ambil `name` pelanggan yang tidak direferensikan oleh pelanggan `id = 2`. Yang `referee_id`-nya NULL **ikut terhitung**.

```sql
SELECT name FROM Customer WHERE referee_id <> 2 OR referee_id IS NULL;
```

**Inti pelajarannya.** Di SQL, `NULL <> 2` bernilai `UNKNOWN` sehingga barisnya terbuang — itulah sebabnya `OR referee_id IS NULL` wajib ditulis.

Di pandas arahnya justru terbalik: `np.nan != 2` bernilai `True`, jadi baris NaN ikut terbawa tanpa Anda minta. Hasilnya kebetulan benar — tapi tetap tulis `.isna()` secara eksplisit, supaya yang terbaca adalah niat Anda, bukan kebetulan.

In [ ]:
customer = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6],
    "name":       ["Sari", "Bima", "Rani", "Dewi", "Agus", "Tono"],
    "referee_id": [np.nan, np.nan, 2.0, 3.0, 2.0, np.nan],
})

print("NaN != 2 ->", customer["referee_id"].ne(2).tolist(), " (True: baris ikut)")
print("NaN <  2 ->", customer["referee_id"].lt(2).tolist(), " (False: baris terbuang)")
print()

mask = (customer["referee_id"] != 2) | customer["referee_id"].isna()
hasil = customer.loc[mask, ["name"]]
show("E2 — hasil", hasil)

assert set(hasil["name"]) == {"Sari", "Bima", "Dewi", "Tono"}

### E3 · LC 1683 — Invalid Tweets · `Easy`

**Butuh join?** Tidak.

**Tugas.** `Tweets(tweet_id, content)`. Ambil `tweet_id` yang `content`-nya lebih dari 15 karakter.

```sql
SELECT tweet_id FROM Tweets WHERE CHAR_LENGTH(content) > 15;
```

In [31]:
tweets = pd.DataFrame({
    "tweet_id": [1, 2, 3, 4],
    "content": ["Halo dunia",
                "Belajar pandas itu menyenangkan sekali",
                "Data engineering",
                "Singkat"],
})

hasil = tweets.loc[tweets["content"].str.len() > 15, ["tweet_id"]]
show("E3 — hasil", hasil)

assert hasil["tweet_id"].tolist() == [2, 3]

=== E3 — hasil ===


,tweet_id
1,2
2,3


In [32]:
tweets.loc[tweets["content"].str.len() > 15]

,tweet_id,content
1,2,Belajar pandas itu menyenangkan sekali
2,3,Data engineering


### E4 · LC 1378 — Replace Employee ID With The Unique Identifier · `Easy`

**Butuh join?** Ini **lookup**, bukan join.

**Tugas.** `Employees(id, name)` dan `EmployeeUNI(id, unique_id)`. Tampilkan `unique_id` dan `name` tiap karyawan; yang tidak punya padanan tetap muncul dengan nilai kosong.

```sql
SELECT eu.unique_id, e.name
FROM Employees e LEFT JOIN EmployeeUNI eu ON e.id = eu.id;
```

**Kenapa ini bukan join sungguhan.** Kunci `id` di `EmployeeUNI` **unik**. Artinya tiap baris kiri mencari **paling banyak satu** baris kanan — itu definisi kamus, bukan definisi join. Dan pandas punya alat khusus untuk kamus:

```python
lookup = employee_uni.set_index("id")["unique_id"]   # kunci -> nilai
employees["id"].map(lookup)                          # NaN kalau tidak ketemu
```

**Tiga sifat `map` yang layak dihafal:**

1. **Bentuknya dijamin.** Output selalu sepanjang input. Jumlah baris tidak mungkin berubah — jadi tidak ada lagi ritual `assert len(hasil) == len(kiri)` setelah merge.
2. **Nilai hilang jadi NaN.** Persis seperti `LEFT JOIN`.
3. **Gagal berisik.** Kalau kunci kanan ternyata tidak unik, `map` melempar error, sementara `merge` menggandakan baris tanpa bilang apa-apa. Sel di bawah membuktikannya.

In [ ]:
employees = pd.DataFrame({
    "id":   [1, 7, 11, 90, 3],
    "name": ["Alice", "Bob", "Meir", "Winston", "Jonathan"],
})
employee_uni = pd.DataFrame({
    "id":        [3, 11, 90],
    "unique_id": [1, 2, 3],
})

lookup = employee_uni.set_index("id")["unique_id"]
show("Tabel lookup (Series ber-index)", lookup)

hasil = pd.DataFrame({
    "unique_id": employees["id"].map(lookup),
    "name":      employees["name"],
})
show("E4 — hasil", hasil)

print("baris kiri:", len(employees), "| baris hasil:", len(hasil), "-> dijamin sama oleh map")
assert len(hasil) == len(employees)
assert hasil["unique_id"].isna().sum() == 2

In [ ]:
# Bukti sifat #3: map gagal berisik saat kunci kanan tidak unik.
uni_rusak = pd.DataFrame({"id": [3, 3, 11], "unique_id": [1, 99, 2]})   # id=3 duplikat

try:
    employees["id"].map(uni_rusak.set_index("id")["unique_id"])
    print("tidak ada error (tidak diharapkan)")
except Exception as e:
    print(f"map -> {type(e).__name__}: {e}")

print()
# Bandingkan: merge akan menghasilkan 6 baris dari 5 baris kiri, tanpa peringatan apa pun.
diam_diam = _merge_asli(employees, uni_rusak, on="id", how="left")
print("merge -> tidak ada error, tapi baris kiri", len(employees),
      "menjadi", len(diam_diam), "baris hasil")
display(diam_diam)

**Kalau kolom yang ditarik lebih dari satu**, `map` per kolom jadi bertele-tele. Gunakan `reindex` — versi lookup untuk baris utuh:

```python
kanan.set_index("k").reindex(kiri["k"])
```

`reindex` menyusun ulang tabel kanan agar **urutan dan panjangnya persis mengikuti** daftar kunci di kiri, mengisi NaN untuk yang tidak ketemu. Itu tepat definisi `LEFT JOIN` untuk kunci kanan yang unik — hanya saja dinyatakan sebagai penyelarasan index, bukan sebagai join.

In [ ]:
uni_lengkap = pd.DataFrame({
    "id":        [3, 11, 90],
    "unique_id": [1, 2, 3],
    "kantor":    ["Malang", "Bandung", "Jakarta"],
})

tarik = uni_lengkap.set_index("id").reindex(employees["id"]).reset_index(drop=True)
hasil2 = employees.copy()
hasil2[["unique_id", "kantor"]] = tarik[["unique_id", "kantor"]].to_numpy()
show("E4 — varian reindex (menarik dua kolom sekaligus)", hasil2)

assert hasil2.loc[hasil2["name"] == "Meir", "kantor"].item() == "Bandung"
assert hasil2["kantor"].isna().sum() == 2

### E5 · LC 1581 — Customer Who Visited but Did Not Make Any Transactions · `Easy`

**Butuh join?** Ini **uji keanggotaan** (anti-join), bukan join.

**Tugas.** `Visits(visit_id, customer_id)` dan `Transactions(transaction_id, visit_id, amount)`. Untuk tiap pelanggan, hitung berapa kali ia berkunjung tanpa bertransaksi.

```sql
SELECT customer_id, COUNT(*) AS count_no_trans
FROM Visits WHERE visit_id NOT IN (SELECT visit_id FROM Transactions)
GROUP BY customer_id;
```

**Ini kasus di mana `merge` justru merepotkan.** Satu kunjungan bisa punya banyak transaksi, jadi `merge(how="left")` menggandakan baris kunjungan — lalu Anda harus hati-hati agar penggandaan itu tidak ikut terhitung. Padahal pertanyaan sebenarnya cuma **ya/tidak**: apakah `visit_id` ini muncul di tabel transaksi?

`isin` menjawab persis pertanyaan itu, dan sama sekali kebal terhadap duplikat di sisi kanan.

**Varian yang berguna:** kalau yang dibutuhkan bukan ya/tidak melainkan *berapa banyak*, resepnya adalah **agregasi dulu, baru `map`**. Agregasi membuat kunci kanan menjadi unik, sehingga `map` kembali aman.

In [12]:
visits = pd.DataFrame({
    "visit_id":    [1, 2, 4, 5, 5, 6, 7, 8],
    "customer_id": [23, 9, 30, 54, 54, 96, 54, 54],
})
transactions = pd.DataFrame({
    "transaction_id": [2, 3, 9, 12],
    "visit_id":       [5, 5, 5, 1],
    "amount":         [310, 300, 200, 910],
})

# --- Cara 1: anti-join = ~isin
tanpa_trx = visits[~visits["visit_id"].isin(transactions["visit_id"])]
show("Kunjungan tanpa transaksi", tanpa_trx)


=== Kunjungan tanpa transaksi ===


,visit_id,customer_id
1,2,9
2,4,30
5,6,96
6,7,54
7,8,54


In [44]:
hasil = (tanpa_trx.groupby("customer_id", as_index=False)
                  .size()
                  .rename(columns={"size": "count_no_trans"}))
show("E5 — hasil", hasil)

=== E5 — hasil ===


,customer_id,count_no_trans
0,9,1
1,30,1
2,54,2
3,96,1


In [ ]:

# --- Cara 2: agregasi dulu (kunci jadi unik), baru map
n_trx = transactions["visit_id"].value_counts()          # visit_id -> jumlah transaksi
v = visits.assign(n_trx=visits["visit_id"].map(n_trx).fillna(0).astype(int))
show("Kunjungan + jumlah transaksinya", v)

hasil2 = (v[v["n_trx"] == 0]
          .groupby("customer_id", as_index=False)
          .size()
          .rename(columns={"size": "count_no_trans"}))

pd.testing.assert_frame_equal(hasil, hasil2)
assert hasil.set_index("customer_id")["count_no_trans"].to_dict() == {9: 1, 30: 1, 54: 2, 96: 1}
print("kedua cara identik.")

=== E5 — hasil ===


,customer_id,count_no_trans
0,9,1
1,30,1
2,54,2
3,96,1



=== Kunjungan + jumlah transaksinya ===


,visit_id,customer_id,n_trx
0,1,23,1
1,2,9,0
2,4,30,0
3,5,54,3
4,5,54,3
5,6,96,0
6,7,54,0
7,8,54,0



kedua cara identik.


### E6 · LC 197 — Rising Temperature · `Easy`

**Butuh join?** SQL menuliskannya sebagai self-join. pandas tidak memerlukannya.

**Tugas.** `Weather(id, recordDate, temperature)`. Ambil `id` hari yang suhunya lebih tinggi daripada hari sebelumnya — persisnya tanggal kemarin, bukan sekadar baris sebelumnya.

```sql
SELECT w1.id FROM Weather w1 JOIN Weather w2
  ON DATEDIFF(w1.recordDate, w2.recordDate) = 1
WHERE w1.temperature > w2.temperature;
```

**Kenapa SQL memakai join padahal datanya satu tabel.** Karena SQL tidak punya konsep "baris sebelumnya" tanpa window function — satu-satunya cara menaruh dua baris berdampingan adalah menempelkan tabel ke dirinya sendiri. pandas punya `shift()`, jadi self-join-nya tidak pernah perlu ada.

**Jebakan besarnya.** `shift(1)` mengambil **baris sebelumnya**, dan baris sebelumnya belum tentu **hari sebelumnya**. Dua langkah ini tidak bisa ditawar:

1. `sort_values` dulu — `shift` tidak tahu apa-apa soal urutan.
2. Verifikasi selisihnya benar-benar 1 hari — jangan berasumsi.

In [ ]:
weather = pd.DataFrame({
    "id":          [1, 2, 3, 4, 5],
    "recordDate":  pd.to_datetime(["2015-01-01", "2015-01-02", "2015-01-03",
                                   "2015-01-06", "2015-01-07"]),   # 04-05 bolong
    "temperature": [10, 25, 20, 30, 35],
})

w = weather.sort_values("recordDate").reset_index(drop=True)
w["suhu_kemarin"]    = w["temperature"].shift(1)
w["tanggal_kemarin"] = w["recordDate"].shift(1)
w["selisih_hari"]    = w["recordDate"] - w["tanggal_kemarin"]
show("Weather + kolom lag", w)

naik = (w["selisih_hari"] == pd.Timedelta(days=1)) & (w["temperature"] > w["suhu_kemarin"])
hasil = w.loc[naik, ["id"]]
show("E6 — hasil (dengan verifikasi jarak)", hasil)

naif = w.loc[w["temperature"] > w["suhu_kemarin"], ["id"]]
show("E6 — versi naif (SALAH: id=4 lolos padahal melompat 3 hari)", naif)

assert hasil["id"].tolist() == [2, 5]
assert naif["id"].tolist() == [2, 4, 5]

### E7 · LC 577 — Employee Bonus · `Easy`

**Butuh join?** Lookup lagi — kunci `empId` di tabel `Bonus` unik.

**Tugas.** `Employee(empId, name, supervisor, salary)` dan `Bonus(empId, bonus)`. Tampilkan nama dan bonus karyawan yang bonusnya di bawah 1000 **atau** tidak punya bonus sama sekali.

```sql
SELECT e.name, b.bonus
FROM Employee e LEFT JOIN Bonus b ON e.empId = b.empId
WHERE b.bonus < 1000 OR b.bonus IS NULL;
```

**Kontras penting dengan E2.** Di E2, operator `!=` **memasukkan** NaN. Di sini `<` **membuang** NaN (`np.nan < 1000` bernilai `False`). Jadi tidak ada aturan tunggal "pandas menyimpan NaN" — perilakunya bergantung pada operator.

Aturan praktisnya: setiap kali kolom hasil lookup muncul di dalam filter, berhentilah sejenak dan tanyakan secara sadar apa yang harus terjadi pada baris NaN — lalu tuliskan `.isna()` secara eksplisit.

In [ ]:
employee = pd.DataFrame({
    "empId":      [3, 1, 2, 4],
    "name":       ["Brad", "John", "Dan", "Thomas"],
    "supervisor": [np.nan, 3.0, 3.0, 2.0],
    "salary":     [4000, 1000, 2000, 4000],
})
bonus = pd.DataFrame({"empId": [2, 4], "bonus": [500, 2000]})

e = employee.assign(bonus=employee["empId"].map(bonus.set_index("empId")["bonus"]))
show("Employee + bonus hasil lookup", e)

mask = (e["bonus"] < 1000) | e["bonus"].isna()
hasil = e.loc[mask, ["name", "bonus"]]
show("E7 — hasil", hasil)

assert set(hasil["name"]) == {"Brad", "John", "Dan"}

### E8 · LC 596 — Classes With at Least 5 Students · `Easy`

**Butuh join?** Tidak.

**Tugas.** `Courses(student, class)`. Ambil nama kelas dengan minimal 5 siswa berbeda.

```sql
SELECT class FROM Courses GROUP BY class HAVING COUNT(DISTINCT student) >= 5;
```

**Yang perlu dihafal.** `WHERE` menyaring **sebelum** agregasi; `HAVING` menyaring **sesudah**. Di pandas tidak ada dua kata kunci berbeda — keduanya boolean mask biasa, yang membedakan hanya posisinya relatif terhadap `.groupby()`.

In [ ]:
courses = pd.DataFrame({
    "student": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "A"],
    "class":   ["Math"]*6 + ["Biology", "Computer", "Math", "Math"],
})

per_kelas = (courses.groupby("class", as_index=False)["student"]
                    .nunique()
                    .rename(columns={"student": "n_siswa"}))
show("Jumlah siswa unik per kelas", per_kelas)

hasil = per_kelas.loc[per_kelas["n_siswa"] >= 5, ["class"]]   # HAVING
show("E8 — hasil", hasil)

assert hasil["class"].tolist() == ["Math"]

### E9 · LC 1661 — Average Time of Process per Machine · `Easy`

**Butuh join?** Ini **reshape**, bukan join.

**Tugas.** `Activity(machine_id, process_id, activity_type, timestamp)` dengan `activity_type` bernilai `'start'`/`'end'`. Hitung rata-rata durasi proses per mesin, bulatkan 3 desimal.

```sql
SELECT machine_id, ROUND(AVG(t_end - t_start), 3) AS processing_time
FROM (
  SELECT machine_id, process_id,
         MAX(CASE WHEN activity_type = 'start' THEN timestamp END) AS t_start,
         MAX(CASE WHEN activity_type = 'end'   THEN timestamp END) AS t_end
  FROM Activity GROUP BY machine_id, process_id
) s GROUP BY machine_id;
```

**Godaan yang harus dilawan.** Refleks pertama biasanya: pisahkan baris `start` dan `end` jadi dua DataFrame, lalu join keduanya. Itu berhasil — tapi ia salah membaca situasinya. `start` dan `end` bukan dua tabel berbeda; keduanya **baris yang sama dari tabel yang sama**, hanya berbeda label. Yang dibutuhkan adalah memutar sumbu, bukan menyambung tabel.

`.unstack()` melakukan tepat itu: ia memindahkan satu level index menjadi kolom.

In [33]:
activity = pd.DataFrame({
    "machine_id":    [0, 0, 0, 0, 1, 1, 1, 1],
    "process_id":    [0, 0, 1, 1, 0, 0, 1, 1],
    "activity_type": ["start", "end", "start", "end"] * 2,
    "timestamp":     [0.712, 1.520, 3.140, 4.120, 0.550, 1.550, 0.430, 1.420],
})

lebar = (activity.set_index(["machine_id", "process_id", "activity_type"])["timestamp"]
                 .unstack("activity_type"))          # level terakhir naik jadi kolom
lebar["durasi"] = lebar["end"] - lebar["start"]
show("Setelah unstack (long -> wide)", lebar)

hasil = (lebar.groupby("machine_id")["durasi"]
              .mean().round(3)
              .rename("processing_time")
              .reset_index())
show("E9 — hasil", hasil)

assert hasil["processing_time"].tolist() == [0.894, 0.995]

=== Setelah unstack (long -> wide) ===


activity_type           end  start  durasi
machine_id process_id                     
0          0           1.52  0.712   0.808
           1           4.12  3.140   0.980
1          0           1.55  0.550   1.000
           1           1.42  0.430   0.990


=== E9 — hasil ===


,machine_id,processing_time
0,0,0.894
1,1,0.995


In [35]:
activity.set_index(["machine_id", "process_id", "activity_type"])["timestamp"].unstack("activity_type")

activity_type           end  start
machine_id process_id             
0          0           1.52  0.712
           1           4.12  3.140
1          0           1.55  0.550
           1           1.42  0.430

In [36]:
activity.set_index(["machine_id", "process_id", "activity_type"])["timestamp"]

machine_id  process_id  activity_type
0           0           start            0.712
                        end              1.520
            1           start            3.140
                        end              4.120
1           0           start            0.550
                        end              1.550
            1           start            0.430
                        end              1.420
Name: timestamp, dtype: float64

In [37]:
lebar["durasi"] = lebar["end"] - lebar["start"]
show("Setelah unstack (long -> wide)", lebar)

=== Setelah unstack (long -> wide) ===


activity_type           end  start  durasi
machine_id process_id                     
0          0           1.52  0.712   0.808
           1           4.12  3.140   0.980
1          0           1.55  0.550   1.000
           1           1.42  0.430   0.990

---
## Kapan `merge` memang tetap diperlukan

Batasan di notebook ini bersifat pedagogis, bukan anjuran permanen. Ada empat situasi di mana `merge` adalah jawaban yang benar, dan memaksakan `map` justru membuat kode lebih buruk:

**1. Relasi many-to-many.** Kalau satu baris kiri harus dipasangkan dengan **beberapa** baris kanan sekaligus dan semuanya harus muncul, jumlah baris output memang wajib bertambah. `map` secara definisi tidak bisa melakukannya — ia menjamin panjang output = panjang input, dan di sini jaminan itu justru yang salah.

**2. Kunci majemuk.** `isin` hanya bekerja pada satu kolom. Untuk mencocokkan pasangan `(player_id, tanggal)` seperti pada soal retensi, `merge` adalah alat yang tepat. (Alternatifnya `MultiIndex.isin` — bisa, tapi jarang lebih terbaca.)

**3. `FULL OUTER JOIN`.** Kalau baris yang hanya ada di sisi kanan juga harus muncul, `map`/`reindex` tidak cukup: keduanya berangkat dari daftar kunci di sisi kiri dan tidak pernah melihat kunci yang eksklusif milik kanan.

**4. Non-equi join dan `merge_asof`.** Pencocokan berdasarkan rentang atau "waktu terdekat sebelumnya" memang keluarga `merge`, dan tidak ada padanan yang lebih sederhana.

**Aturan praktis yang bisa dibawa pulang:**

> Sebelum menulis `merge`, tanyakan: **apakah jumlah baris saya seharusnya berubah?**
>
> - **Tidak** → yang Anda butuhkan adalah lookup. Pakai `map` atau `reindex`; keduanya menjamin hal itu dan berteriak kalau asumsinya salah.
> - **Berkurang** → itu penyaringan. Pakai `isin`.
> - **Bertambah** → barulah `merge`, dan tulis `validate=` supaya penambahannya sesuai yang Anda maksud.

## Ringkasan

| Soal | Di notebook utama | Di sini | Pola sebenarnya |
|---|---|---|---|
| E1 | mask | mask | filtering |
| E2 | mask | mask | semantik NULL |
| E3 | `.str` | `.str` | operasi string |
| E4 | `merge(how="left")` | `.map()` / `.reindex()` | **lookup** |
| E5 | `merge` + `indicator` | `~isin()` / agregasi + `map` | **keanggotaan** |
| E6 | `shift` | `shift` | perbandingan antar-baris |
| E7 | `merge(how="left")` | `.map()` | **lookup** |
| E8 | `groupby` | `groupby` | agregasi + `HAVING` |
| E9 | `pivot_table` | `.unstack()` | **reshape** |

Dari sembilan soal, hanya tiga yang memakai `merge` di notebook utama — dan ketiganya ternyata **bukan** join sungguhan. Itulah poin utama latihan ini.

**Latihan lanjutan.** Coba terapkan batasan yang sama ke bagian Medium. Sebagian akan menyerah dengan mudah (M2 sudah memakai `value_counts`, M9 dan M10 tidak butuh join sama sekali). Tetapi M1 (cross join) dan M4 (non-equi join) akan **melawan** — dan justru di titik perlawanan itulah Anda belajar kapan `merge` benar-benar tak tergantikan.

In [ ]:
# Kembalikan merge agar sel-sel eksperimen setelah notebook ini tetap bisa jalan.
pd.merge = _merge_asli
print("pd.merge dikembalikan.")